In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

In [ ]:
df = spark.read.json("/home/jovyan/notebooks/mojfolder/transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

In [ ]:
df.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

1.

In [ ]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round, window, min as _min, desc

gdanskmin = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_kwota")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_kwota"
    )
    .orderBy("srednia_kwota")
)
gdanskmin.show(1)

2.

In [ ]:
percat= (
    df.groupBy(
        window("timestamp", "30 minutes"),
        "category"
    )
    .agg(
        count("tx_id").alias("liczba_transakcji")
    )
    .filter(col("window.start") == "2026-04-12 09:00:00")
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "category",
        "liczba_transakcji"
    )
    .orderBy("category")
)
percat.show()

3.

In [ ]:
transactions_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_transakcji"),
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_transakcji",
        "suma_PLN"
    )
    .orderBy(desc("liczba_transakcji"))
)

transactions_15min.show(1, truncate=False)